# Spark NLP{background-color="black"}

*State of the art Natural Language Processing, natively on Apache Spark*

- Open source (Apache 2.0) by John Snow Labs, **built on top of Spark MLlib**
- Thousands of pretrained models and pipelines on the [Models Hub](https://sparknlp.org/models), 200+ languages
- Transformers (BERT, RoBERTa, DeBERTa, ...), embeddings, NER, classification, translation, speech, vision, **LLMs**
- Scales from a laptop (`local[*]`) to a cluster without changing code
- Current release: **7.0.1** (7.0 line since Sept 2026), supports **Spark 4.x** / Scala 2.13 / Java 17

## Where we are

```{mermaid}
%%| echo: false
%%{init: {"flowchart": {"nodeSpacing": 40, "rankSpacing": 100}}}%%
flowchart LR
  A["MLlib<br/>Spark NLP<br/><b>Executor JVM</b>"] --> B["Hugging Face<br/>UDF<br/><b>Python worker</b>"] --> C["Served LLM<br/><b>Separate server</b>"]

  classDef jvm fill:#dcefe5,stroke:#27805a,stroke-width:3px,color:#173b2c,font-size:26px
  classDef python fill:#e5edf9,stroke:#5275a5,stroke-width:3px,color:#263b59,font-size:26px
  classDef remote fill:#fff0d9,stroke:#b87922,stroke-width:3px,color:#563b17,font-size:26px
  class A jvm
  class B python
  class C remote
```

::: {.fragment}
**Key distinction:** the model runs in the executor JVM, a Python worker, or a separate server.
:::

## Core concepts: Annotators

:::: {.columns}

::: {.fragment .column width="50%"}
An **annotator** is an MLlib stage that reads annotation columns and writes a new one.

- `AnnotatorApproach` → an **Estimator** (it must be trained: `fit()`)
- `AnnotatorModel` → a **Transformer** (already trained, often `.pretrained()`)
- Every annotator declares `setInputCols([...])` / `setOutputCol(...)` and the **types** it accepts (`DOCUMENT`, `TOKEN`, `WORD_EMBEDDINGS`, `CATEGORY`, ...)
:::

::: {.fragment .column width="50%"}
**Special transformers**

- `DocumentAssembler`: raw string column → `DOCUMENT` annotation (entry point)
- `Finisher` / `EmbeddingsFinisher`: annotations → plain Spark columns (exit point)
- `LightPipeline`: runs a fitted pipeline **on the driver**, without a DataFrame → low latency for single strings
:::
::::

### The Annotation schema

Each annotation column is an `array<struct>`:

| field | meaning |
|---|---|
| `annotatorType` | `document`, `token`, `chunk`, `category`, ... |
| `begin`, `end` | character offsets in the original text |
| `result` | the value (token, label, entity, generated text) |
| `metadata` | map with extra info (confidence, sentence id, ...) |
| `embeddings` | vector, when the annotator produces embeddings |

::: {.fragment}
Keeping **offsets** is what makes Spark NLP suitable for highlighting, redaction (PII) and traceable extraction.
:::

## Setup with Docker

:::: {.columns}
::: {.column width="55%"}
`sparknlp/Dockerfile`
```Dockerfile
FROM python:3.12-bookworm

# Spark 4 requires Java 17
RUN apt-get update && \
    apt-get install -y --no-install-recommends openjdk-17-jre-headless && \
    apt-get clean && rm -rf /var/lib/apt/lists/*

ENV JAVA_HOME=/usr/lib/jvm/java-17-openjdk-amd64
ENV PATH=$JAVA_HOME/bin:$PATH

# See https://sparknlp.org/docs/en/install#python
# spark-nlp 7.0.x + pyspark 4.0.1 (and later validated 4.x)
# setuptools: distutils shim still imported by spark-nlp on Python 3.12
RUN pip install --no-cache-dir numpy setuptools spark-nlp==7.0.1 pyspark==4.0.1

WORKDIR /usr/src/app
ENTRYPOINT ["python3"]
CMD ["example.py"]
```
:::

::: {.column width="45%"}
Build
```bash
cd sparknlp
docker build -t tap:sparknlp .
```

Run with **persistent caches** (jars and models are downloaded once)
```bash
docker run --rm -it -p 4040:4040 \
  -e SPARK_LOG_LEVEL=WARN \
  -v sparknlp-ivy-cache:/root/.ivy2 \
  -v sparknlp-models:/root/cache_pretrained \
  -v ./src:/usr/src/app \
  -v ./models:/models \
  tap:sparknlp example.py
```

::: {.fragment}
`sparknlp.start()` creates the `SparkSession` and pulls the right `spark-nlp_2.13` jar for your Spark version.
:::
:::
::::

## Pretrained pipelines

`sparknlp/src/example.py`

In [ ]:
import sparknlp
from sparknlp.pretrained import PretrainedPipeline

spark = sparknlp.start(memory="8G")        # gpu=True, apple_silicon=True are also available
print("Spark NLP", sparknlp.version(), "- Spark", spark.version)

pipeline = PretrainedPipeline("recognize_entities_dl", lang="en")

text = """The Mona Lisa is a 16th century oil painting created by Leonardo.
It's held at the Louvre in Paris."""

result = pipeline.annotate(text)          # LightPipeline under the hood
for k in ["token", "ner", "entities"]:
    print(k, "->", result[k])

::: {.fragment}
- `annotate(str)` → dictionary, runs on the driver (great for tests and APIs)
- `transform(df)` → distributed, runs on executors (great for millions of documents)
:::

::: {.fragment}
```python
df = spark.createDataFrame([(text,)], ["text"])
pipeline.transform(df).selectExpr("explode(entities.result) as entity").show()
```
:::

## Building your own pipeline: transformer sentiment

`sparknlp/src/sentiment.py` — model card: [twitter_roberta_base_sentiment_latest](https://sparknlp.org/2025/05/22/twitter_roberta_base_sentiment_latest_en.html)

In [ ]:
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, RoBertaForSequenceClassification
from pyspark.ml import Pipeline

document = DocumentAssembler().setInputCol("text").setOutputCol("document")
token = Tokenizer().setInputCols(["document"]).setOutputCol("token")
classifier = (RoBertaForSequenceClassification
              .pretrained("twitter_roberta_base_sentiment_latest", "en")
              .setInputCols(["document", "token"])
              .setOutputCol("class")
              .setBatchSize(16))

pipeline = Pipeline(stages=[document, token, classifier])

data = spark.createDataFrame([["This lesson is not boring at all"],
                              ["I'm mad, why am I here?"]]).toDF("text")
model = pipeline.fit(data)          # no training: all stages are already models
model.transform(data).select("text", "class.result").show(truncate=False)

::: {.fragment}
Same abstractions as MLlib: the result is a `PipelineModel` you can `save()` and load in a **Structured Streaming** job reading from Kafka.
:::

::: {.fragment}
Compare with the TF-IDF model of the previous block on *"This lesson is not boring at all"*.
:::

## Under the hood: inference engines

A Spark NLP transformer annotator is **a model file + a runtime embedded in the JVM**.

| Engine | Model format | Typical use in Spark NLP |
|---|---|---|
| TensorFlow | SavedModel | historical default (older models) |
| **ONNX Runtime** | `.onnx` | most transformer models since 5.x, fast on CPU |
| **OpenVINO** | IR (`.xml` + `.bin`) | Intel CPUs/GPUs |
| **llama.cpp** | **`.gguf`** | LLMs with `AutoGGUFModel`, `AutoGGUFEmbeddings`, `AutoGGUFVisionModel` |

::: {.fragment}
Key idea: **the model is detached from the framework that trained it.** You train/fine-tune in PyTorch, then you *export* and run it anywhere — including inside a Spark executor, with no Python and no PyTorch installed.
:::

## ONNX: Open Neural Network Exchange

:::: {.columns}

::: {.fragment .column width="50%"}
**What it is**

- An open standard (Linux Foundation AI) to describe a model as a **computation graph**
- The `.onnx` file contains: graph of **operators** (MatMul, Softmax, Attention, ...), **weights**, input/output signatures
- **Opset** = versioned set of operators → compatibility contract between exporter and runtime
- Born in 2017 (Microsoft + Facebook) to stop the "framework lock-in"
:::

::: {.fragment .column width="50%"}
**ONNX Runtime**

- High performance engine in C++, bindings for **Python, Java (JVM!), C#, JavaScript, mobile**
- **Execution providers**: CPU, CUDA, TensorRT, OpenVINO, CoreML, DirectML, ...
- Graph optimizations (operator fusion, constant folding)
- **Quantization** (FP32 → INT8): ~4x smaller, 2-3x faster on CPU, small accuracy loss
:::
::::

::: {.fragment}
```{mermaid}
flowchart LR
  PT[PyTorch] -- export --> O[(model.onnx)]
  TF[TensorFlow] -- export --> O
  SK[scikit-learn] -- skl2onnx --> O
  O --> RJ[ONNX Runtime JVM<br/>Spark NLP]
  O --> RP[ONNX Runtime Python<br/>Pandas UDF]
  O --> RW[ONNX Runtime Web<br/>browser]
  O --> RM[Mobile / Edge]
```
:::

### Step 1 — Export a Hugging Face model to ONNX

[🤗 Optimum](https://huggingface.co/docs/optimum-onnx/) converts a Transformers model (PyTorch) into ONNX.

`sparknlp/export_onnx.sh` runs in a throw-away Python container, so the Spark NLP image stays small:

```bash
MODEL=distilbert/distilbert-base-uncased-finetuned-sst-2-english
docker run --rm -v ./models:/models python:3.12-slim bash -c "
  pip install -q 'optimum[onnx]' onnx onnxruntime torch --extra-index-url https://download.pytorch.org/whl/cpu &&
  optimum-cli export onnx --model $MODEL --task text-classification /models/distilbert_sst2_onnx"
```

::: {.fragment}
```text
models/distilbert_sst2_onnx/
├── config.json            # id2label: {0: NEGATIVE, 1: POSITIVE}
├── model.onnx             # graph + weights (~260 MB)
├── tokenizer.json
├── tokenizer_config.json
└── vocab.txt
```

Open `model.onnx` in [Netron](https://netron.app) to *see* the graph.
:::

### Step 2 — Run it with plain ONNX Runtime (no PyTorch!)

Understand what a runtime does: tokens in, logits out.

In [ ]:
# pip install onnxruntime tokenizers numpy
import numpy as np, onnxruntime as ort
from tokenizers import Tokenizer

path = "../sparknlp/models/distilbert_sst2_onnx"
tok = Tokenizer.from_file(f"{path}/tokenizer.json")
sess = ort.InferenceSession(f"{path}/model.onnx", providers=["CPUExecutionProvider"])

print([i.name for i in sess.get_inputs()], "->", [o.name for o in sess.get_outputs()])

enc = tok.encode("This lesson is not boring at all")
logits = sess.run(None, {"input_ids": np.array([enc.ids]),
                         "attention_mask": np.array([enc.attention_mask])})[0]
probs = np.exp(logits) / np.exp(logits).sum(-1, keepdims=True)
print(dict(zip(["NEGATIVE", "POSITIVE"], probs[0].round(3))))

### Step 3 — Import the ONNX model into Spark NLP

Spark NLP expects the tokenizer files in an `assets/` folder (the last step of `export_onnx.sh` does it):

```text
models/distilbert_sst2_onnx/
├── model.onnx
└── assets/
    ├── vocab.txt          # copied from the export
    └── labels.txt         # NEGATIVE\nPOSITIVE, from config.json id2label
```

`sparknlp/src/onnx_import.py`

In [ ]:
from sparknlp.annotator import DistilBertForSequenceClassification

onnx_dir = "/models/distilbert_sst2_onnx"

# loadSavedModel detects model.onnx and uses the ONNX Runtime engine
clf = (DistilBertForSequenceClassification
       .loadSavedModel(onnx_dir, spark)
       .setInputCols(["document", "token"])
       .setOutputCol("class")
       .setCaseSensitive(False))

# Save it once in Spark NLP format: from now on it is a normal annotator
clf.write().overwrite().save("/models/distilbert_sst2_spark_nlp")

In [ ]:
clf = (DistilBertForSequenceClassification.load("/models/distilbert_sst2_spark_nlp")
       .setInputCols(["document", "token"]).setOutputCol("class"))

pipeline = Pipeline(stages=[document, token, clf])
tweets = spark.createDataFrame([["This lesson is not boring at all"],
                                ["Waiting for the 2 a.m. bus again"]]).toDF("text")
pipeline.fit(tweets).transform(tweets).select("text", "class.result").show(truncate=False)
# +--------------------------------+----------+
# |text                            |result    |
# +--------------------------------+----------+
# |This lesson is not boring at all|[POSITIVE]|   <- 0.998, the TF-IDF model said NEGATIVE
# |Waiting for the 2 a.m. bus again|[NEGATIVE]|
# +--------------------------------+----------+

::: {.fragment}
**What did we gain?**

- Any of the ~1M models on the Hugging Face Hub (with a supported architecture) becomes a **distributed Spark annotator**
- Executors need **no Python, no PyTorch**: just the JVM + ONNX Runtime
- The same `.onnx` file can be quantized and reused in the HF/Python world, in the browser or on mobile
:::

::: {.fragment}
**Supported import architectures**: BERT, DistilBERT, RoBERTa, XLM-R, DeBERTa, ALBERT, CamemBERT, MPNet, E5, BGE, Whisper, T5, BART, ... → see the [import notebooks](https://github.com/JohnSnowLabs/spark-nlp/tree/master/examples/python/transformers/onnx).
:::

## The same idea for LLMs: GGUF and llama.cpp

:::: {.columns}

::: {.fragment .column width="50%"}
ONNX is great for encoders (classification, NER, embeddings). For **generative LLMs** the de facto portable format is **GGUF**:

- single file: weights (often **quantized** to 4-8 bit) + tokenizer + metadata
- executed by **[llama.cpp](https://github.com/ggml-org/llama.cpp)**, a C/C++ runtime that runs on CPU, Apple Metal, CUDA, Vulkan
- Spark NLP embeds llama.cpp with `AutoGGUFModel`
:::

::: {.fragment .column width="50%"}
`sparknlp/src/gguf.py`
```python
from sparknlp.annotator import AutoGGUFModel

llm = (AutoGGUFModel
       .loadSavedModel("/models/qwen2.5-0.5b-instruct-q4_k_m.gguf", spark)
       .setInputCols(["document"])
       .setOutputCol("completions")
       .setBatchSize(4)
       .setNPredict(64)        # max new tokens
       .setTemperature(0.2)
       .setNGpuLayers(0))      # CPU only

prompts = spark.createDataFrame(
    [["Summarize in 5 words: Spark is a unified engine for large-scale data analytics."]]
).toDF("text")
Pipeline(stages=[document, llm]).fit(prompts).transform(prompts) \
    .select("completions.result").show(truncate=False)
```
:::
::::

## ...but should an LLM live inside the executors?

:::: {.columns}

::: {.fragment .column width="50%"}
**Small models (≤ 500M params)**: yes, it works well

- encoders in ONNX: ~100-500 MB, ms per sentence on CPU
- data locality, no network hop, scales with executors
:::

::: {.fragment .column width="50%"}
**LLMs (1B-70B params)**: usually no

- a 7B Q4 model ≈ **4-5 GB RAM per executor** (× every executor, × every restart)
- each executor loads it again → minutes of warm-up; tasks retried = model reloaded
- no **continuous batching**, no shared KV cache across executors
- GPUs are scheduled per *task*, not shared smartly among concurrent requests
- the model's lifecycle (versions, updates) is tied to the Spark job
:::
::::

::: {.fragment}
**Decouple**: the model runs in a dedicated **inference server** (exposing an OpenAI-compatible API), Spark sends **batched, concurrent** requests.

👉 Next notebook: **[Local Inference: llama.cpp, Ollama, vLLM](Local%20Inference.ipynb)**
:::

## Choosing where the model runs

| | Spark NLP (JVM) | HF in Pandas UDF | Served model (HTTP) |
|---|---|---|---|
| Model location | inside executor JVM | Python worker per executor | separate server/pod |
| Formats | ONNX, OpenVINO, TF, GGUF | PyTorch, ONNX (optimum) | GGUF, safetensors |
| Best for | encoders, NER, embeddings at scale | any HF model, quick prototypes | LLMs, GPUs, shared by many apps |
| Scaling | with executors | with executors | independently (replicas, GPUs) |
| Main cost | model download/load per executor | Python ↔ JVM Arrow transfer | network + server capacity |

## Exercise

1. Export `cardiffnlp/twitter-roberta-base-sentiment-latest` (or an **Italian** model, e.g. a `neuraly/bert-base-italian-cased-sentiment`) to ONNX and import it with `RoBertaForSequenceClassification` / `BertForSequenceClassification.loadSavedModel`
2. Quantize it to INT8 (`optimum-cli onnxruntime quantize --avx2`) and compare **size**, **time per 1,000 tweets** and **accuracy** on Sentiment140
3. Write the annotated tweets to Parquet/Iceberg: which columns would you keep for the course project?

# Biblio

* https://sparknlp.org/docs/en/quickstart
* https://sparknlp.org/docs/en/install
* https://sparknlp.org/docs/en/annotators
* https://sparknlp.org/models
* https://github.com/JohnSnowLabs/spark-nlp/tree/master/examples/python/transformers/onnx
* https://onnx.ai/ and https://onnxruntime.ai/docs/
* https://huggingface.co/docs/optimum-onnx/
* https://netron.app
* https://github.com/ggml-org/llama.cpp
* https://huggingface.co/docs/hub/gguf